# **Continuous underway surface measurements along the Polarstern PS124 Track**

*How do surface ocean conditions change continuously along the Polarstern's cruise track?*

For an interactive version of this page please visit the Google Colab: [Open in Google Colab](https://colab.research.google.com/drive/10LZYrXn749aanuHI9whvaV8vZKPel-Qy#scrollTo=0BN_ZWoyxPkz)

*(To open link in new tab press Ctrl + click)*

Alternatevly this notebook can be opened with Binder by following the link: [Continuous underway surface measurements along the Polarstern PS124 Track](https://mybinder.org/v2/gh/s4oceanice/literacy.s4oceanice/main?urlpath=%2Fdoc%2Ftree%2Fnotebooks_binder%2FPolarstern_PS124_route.ipynb)

**Notebook objectives**

This notebook provides an interactive environment to explore continuous underway surface measurements collected during the Polarstern PS124 cruise, using data served through the ERDDAP service hosted by EMODnet Physics. Users can:

*  Select a surface variable (e.g., temperature, salinity, dissolved oxygen, chlorophyll-a) for visualization.
*  View the full cruise track on an interactive map, colored according to the selected variable's value.
*  Explore individual measurement popups showing the variable's value and unit.

**Data sources**

https://erddap.emodnet-physics.eu/erddap/tabledap/ARICE_de_pangaea_dataset961780.html

This dataset is a continuous underway record collected while the ship was in transit.

**How to use this notebook**
1. Run the code cells sequentially from top to bottom.
2. Wait for the remote dataset to be downloaded and processed.
3. Use the available menus, sliders, or map controls to select the variables and periods of interest.
4. Read the interpretation guidance before drawing scientific conclusions from the visualizations.

The notebook retrieves data from remote services. An active internet connection is therefore required.

**Data retrieval and preparation**

The code below imports the libraries, loads the configured ERDDAP table, cleans the observations, selects the shallowest valid station records, and builds the interactive map.

The dropdown menu controls the parameter used to colour the station markers.

- Marker locations correspond to the available coordinates.
- Marker colours represent the selected measurement.
- The legend reports the numerical range.
- Popups provide information for individual observations.
- The line joining stations gives an approximate route sequence.

In [ ]:
# @title
import pandas as pd
import folium
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from IPython.display import display
import ipywidgets as widgets
import numpy as np

DATA_URL = 'https://erddap.emodnet-physics.eu/erddap/tabledap/ARICE_de_pangaea_dataset961780.csv'

def load_and_preprocess_data(url):
    """Loads CTD data and keeps only the surface (shallowest-depth) value per station."""
    try:
        df = pd.read_csv(url, skiprows=[1])

        lat_col = [col for col in df.columns if 'latitude' in col.lower()][0]
        lon_col = [col for col in df.columns if 'longitude' in col.lower()][0]
        df = df.rename(columns={lat_col: 'latitude', lon_col: 'longitude'})

        df = df.dropna(subset=['latitude', 'longitude'])

        for col in df.columns:
            if 'time' not in col.lower():
                converted = pd.to_numeric(df[col], errors='coerce')
                if not converted.isna().all():
                    df[col] = converted

        depth_cols = [col for col in df.columns if 'depth' in col.lower()]
        if depth_cols:
            depth_col = depth_cols[0]
            time_col = [col for col in df.columns if 'time' in col.lower()]
            group_cols = ['latitude', 'longitude'] + (time_col[:1] if time_col else [])
            df = df.sort_values(depth_col).groupby(group_cols, as_index=False).first()

        return df
    except Exception as e:
        print(f"Error loading data from {url}: {e}")
        return pd.DataFrame()

df_arcticnet = load_and_preprocess_data(DATA_URL)
print("Data loading complete.")

VARIABLE_METADATA = {
    "depth": ("Depth", "m"),
    "TEMP": ("Sea water temperature", "°C"),
    "TUR3": ("Light transmission", "%"),
    "FLU2": ("Chlorophyll-a fluorescence", "mg/m3"),
    "PSAL": ("Practical salinity", "PSU"),
    "DENS": ("Sea density [sigma-theta]", "kg/m3"),
    "POTENTIAL_TEMP": ("Sea potential temperature", "°C"),
    "SIGMA_THETA": ("Sea sigma-theta", "kg/m3"),
    "DOX1": ("Dissolved oxygen", "ml/l"),
    "NTRA": ("Nitrate [NO3-N]", "MMole/M3"),
    "LGHT": ("Immersed incoming photosynthetic active radiation", "mE/m2/s"),
    "LGH4": ("Surface incoming photosynthetic active radiation", "umole/[m2/s]"),
    "AMON": ("Absolute salinity", "g/kg"),
}

def get_variable_label(variable_name):
    display_name, unit = VARIABLE_METADATA.get(variable_name, (variable_name, ""))
    return f"{display_name} ({unit})" if unit else display_name

def plot_colored_route(variable_name):
    if variable_name not in df_arcticnet.columns:
        print(f"Variable not found: {variable_name}")
        return

    df = df_arcticnet.dropna(subset=["latitude", "longitude", variable_name])
    if df.empty:
        print("No valid data available for this variable.")
        return

    display_name, unit = VARIABLE_METADATA.get(variable_name, (variable_name, ""))
    label = get_variable_label(variable_name)

    m = folium.Map(location=[df["latitude"].mean(), df["longitude"].mean()], zoom_start=5)

    vmin, vmax = df[variable_name].min(), df[variable_name].max()
    colormap = plt.get_cmap("viridis")
    norm = colors.Normalize(vmin=vmin, vmax=vmax) if vmin != vmax else colors.Normalize(vmin=vmin - 1, vmax=vmax + 1)

    for _, row in df.iterrows():
        val = row[variable_name]
        hex_color = colors.rgb2hex(colormap(norm(val))[:3])
        popup_text = f"<b>{display_name}</b><br>{val:.2f} {unit}" if unit else f"<b>{display_name}</b><br>{val:.2f}"
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5, color=hex_color, fill=True, fill_color=hex_color,
            fill_opacity=0.8, popup=folium.Popup(popup_text, max_width=300)
        ).add_to(m)

    display(m)

    fig, ax = plt.subplots(figsize=(8, 1))
    fig.subplots_adjust(bottom=0.5)
    fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=colormap), cax=ax, orientation="horizontal", label=label)
    plt.show()

numeric_cols = df_arcticnet.select_dtypes(include=[np.number]).columns.tolist()
exclude_keywords = ["latitude", "longitude", "time", "depth"]
vars_available = [c for c in numeric_cols if not any(x in c.lower() for x in exclude_keywords) and not c.upper().endswith("_QC")]

variable_dropdown = widgets.Dropdown(
    options=[(get_variable_label(c), c) for c in vars_available],
    description="Variable:"
)
output_map = widgets.Output()

def on_change(change):
    with output_map:
        output_map.clear_output(wait=True)
        if variable_dropdown.value:
            plot_colored_route(variable_dropdown.value)

variable_dropdown.observe(on_change, "value")
display(variable_dropdown, output_map)
on_change(None)

Data loading complete.


Dropdown(description='Variable:', options=(('Bottle', 'Bottle'), ('Press', 'Press'), ('Sea water temperature (…

Output()

**Interpretation guidance**

The displayed values represent the shallowest retained observations and should not be interpreted as complete profiles. Spatial patterns may be influenced by differences in sampling time, local conditions, measurement availability, and station spacing.

The route line reflects the order of the processed records rather than a high-frequency navigation track. Before publication or reuse, confirm that the ERDDAP source configured in the code corresponds to the intended Polarstern PS124 dataset.

**Technical notes**

The notebook uses `pandas` and `NumPy` for data handling, `folium` for interactive mapping, `Matplotlib` for colour normalization, and `ipywidgets` for the parameter selector.

The list of available parameters is generated from the numerical columns identified after data cleaning. The results therefore depend on the variables and metadata supplied by the remote service.


**Additional resources and acknowledgement**

Main resources used across the notebook series include:

Libraries used:
*  [pandas](https://pandas.pydata.org/)
*  [numpy](https://numpy.org/doc/)
*  [folium](https://python-visualization.github.io/folium/latest/)
*  [matplotlib](https://matplotlib.org/)
*  [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/)


This work has received funding from the European Union Horizon Europe project Ocean-Cryosphere Exchanges in ANtarctica: Impacts on Climate and the Earth System (OCEAN ICE) under grant agreement No. 101060452 (https://doi.org/10.3030/101060452). UK partners are funded by UK Research and Innovation (UKRI) under the UK government's Horizon Europe funding guarantee.

This notebook makes use of data from the ARICE (Arctic Research Icebreaker Consortium) project, hosted via EMODnet Physics (https://www.emodnet-physics.eu).

<center>
  <div style="display: flex; justify-content: center; align-items: flex-start; gap: 20px;">
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/TO-USE-RGB-for-digital-materials-V.png" height="140" style="margin-top: 50px;"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2025/02/UKRI-logo-1.png" height="100"/>
    <img src="https://ocean-ice.eu/wp-content/uploads/2023/06/logo-polar-cluster-2.png" height="100"/>
    <img src="https://emodnet.ec.europa.eu/sites/emodnet.ec.europa.eu/files/public/emodnet_logos/web/EMODnet_standard_colour.png" height="100"/>
  </div>
</center>